# `04-analysis.ipynb`: Analysis of scraped and cleaned data.

## Setup

In [1]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from umpd_crime.constants import MONTHS

from pathlib import Path

PROC_DATA_DIR = Path.cwd().parent / "data" / "processed" 

CUR_TIMESTAMP = time.localtime()
CUR_YEAR = CUR_TIMESTAMP.tm_year
CUR_MONTH = CUR_TIMESTAMP.tm_mon

In [41]:
arrests_and_ci = pd.read_csv(PROC_DATA_DIR / f"merged_arrest_ci_data_2018-{CUR_YEAR}.csv")

In [53]:
disposition_hist = arrests_and_ci['disposition'].value_counts().reset_index()
disposition_hist
disposition_hist['perc'] = disposition_hist['count'] / len(arrests_and_ci['disposition'])
disposition_hist

,disposition,count,perc
0,CBE,6466,0.487228
1,Arrest,3067,0.231105
2,Investigation Pending,1743,0.131339
3,Active/Pending,1417,0.106774
4,Unfounded,240,0.018085
5,Summons Issued,174,0.013111
6,Warrant Issued,124,0.009344
7,Juvenile Arrest,40,0.003014


In [59]:
arrests_and_ci['crime_type'].value_counts().head(15)

crime_type
Injured/Sick Person          2710
Theft                        2146
DWI/DUI                      1343
Assist Other Agency           860
CDS Violation                 635
Other Incident                617
Emergency Petition            529
Vandalism                     504
Suspicious Activity           342
Damage to State Property      261
Trespassing                   184
Burglary                      179
Alcohol Violation             179
Dept Property Damage/Loss     176
Theft from Auto               174
Name: count, dtype: int64

In [70]:
arrests_and_ci['report_date'] = pd.to_datetime(arrests_and_ci['report_date'])
arrests_and_ci['arrest_date'] = pd.to_datetime(arrests_and_ci['arrest_date'])
arrests_and_ci['arrest_date_mo'] = arrests_and_ci['arrest_date'].dt.month_name()
arrests_and_ci['report_date_mo'] = arrests_and_ci['report_date'].dt.month_name()
arrests_and_ci['report_date_mo'].value_counts()

report_date_mo
September    1821
October      1750
April        1270
February     1252
March        1244
November     1152
May          1106
August       1015
January       703
July          703
June          629
December      626
Name: count, dtype: int64

In [72]:
arrests_and_ci[arrests_and_ci['arrest_number'].isna() == False]['arrest_date_mo'].value_counts()

arrest_date_mo
August       325
September    325
March        310
May          310
October      308
July         301
April        298
February     292
January      231
November     226
June         198
December     174
Name: count, dtype: int64

## Uniform Crime Reports

There's the 2023-2026 National Incident-Based Reporting System (NIBRS) and the 2012-2022 Historical Summary Reporting System (SRS).

In [3]:
nibrs = pd.read_csv(PROC_DATA_DIR / "uniform-crime-reports" / f"ucr_2023-{CUR_YEAR}_monthly.csv")

### NIBRS

In [4]:
nibrs['month'] = pd.Categorical(nibrs['month'], categories=MONTHS, ordered=True)
nibrs

,year,crime,crime_type,month,count
0,2025,Drug/Narcotic Violations,crimes_against_property,Jan,1
1,2025,Drug Equipment Violations,crimes_against_property,Jan,2
2,2025,Pornography/Obscene Material,crimes_against_property,Jan,0
3,2025,Prostitution,crimes_against_property,Jan,0
4,2025,Assisting or Promoting Prostitution,crimes_against_property,Jan,0
...,...,...,...,...,...
3019,2026,Simple Assault,crimes_against_society,Dec,0
3020,2026,Intimidation,crimes_against_society,Dec,0
3021,2026,Incest,crimes_against_society,Dec,0
3022,2026,Statutory Rape,crimes_against_society,Dec,0


Let's look at some crimes individually

In [25]:
dui_23_26 = nibrs[nibrs['crime'] == "Driving Under the Influence"].pivot(
  index = 'month',
  columns = 'year',
  values = 'count'
)

dui_23_26_t = dui_23_26.T.reset_index()

dui_23_26[f'2023-{CUR_YEAR}_%_change'] = ((dui_23_26[2026] - dui_23_26[2023]) / dui_23_26[2023]) * 100

In [26]:
dui_23_26

year,2023,2024,2025,2026,2023-2026_%_change
month,,,,,
Jan,5,10,17,10,100.000000
Feb,13,9,11,17,30.769231
Mar,8,9,19,23,187.500000
Apr,7,11,10,25,257.142857
May,11,6,16,40,263.636364
Jun,7,14,9,27,285.714286
Jul,11,26,14,20,81.818182
Aug,10,15,22,42,320.000000
Sep,3,7,11,0,-100.000000


In [35]:
dui_23_26_t

month,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,2023,5,13,8,7,11,7,11,10,3,9,7,6
1,2024,10,9,9,11,6,14,26,15,7,8,10,13
2,2025,17,11,19,10,16,9,14,22,11,16,21,16
3,2026,10,17,23,25,40,27,20,42,0,0,0,0


In [36]:
dui_23_26_t['total'] = dui_23_26_t[MONTHS].sum(axis=1)

In [39]:
dui_23_26_t.T

,0,1,2,3
month,,,,
year,2023,2024,2025,2026
Jan,5,10,17,10
Feb,13,9,11,17
Mar,8,9,19,23
Apr,7,11,10,25
May,11,6,16,40
Jun,7,14,9,27
Jul,11,26,14,20
Aug,10,15,22,42


In [40]:
((204-97)/97) * 100

110.30927835051547